In [ ]:
%pip install pylatexenc
%pip install matplotlib
# %pip install git+https://github.com/It4innovations/quantum-as-a-service.git@main

In [9]:
import os

os.environ["QPROVIDER_LOGLEVEL"] = "INFO"

# Connect to system

In [1]:
# Setup Lexis session
from py4lexis.session import LexisSession

lexis_session = LexisSession()

Welcome to the Py4Lexis (v5.5.0)!
Proceeding login via LEXIS login page...
Open provided url in your browser, please: https:/aai.lexis.tech/auth/realms/LEXIS_AAI_v2/protocol/openid-connect/auth/device?user_code=VIXA-ABIK
Check if user is logged in: Pending.                                            
Check if user is logged in: OK!
You have been successfully logged in LEXIS session.


In [11]:
token = lexis_session.get_access_token()
lexis_project = "vlq_demo_project"
resource_name = "VLQ-CZ"

In [12]:
from qaas.client import QProvider, QBackend, QJob

provider = QProvider(token, lexis_project)
# QBackend
backend: QBackend = provider.get_backend(resource_name)

print(f"Qubit: {backend.architecture.qubits}")

print(f"Gates: {backend.architecture.gates.keys()}")

Qubit: ['QB1', 'QB2', 'QB3', 'QB4', 'QB5', 'QB6', 'QB7', 'QB8', 'QB9', 'QB10', 'QB11', 'QB12', 'QB13', 'QB14', 'QB15', 'QB16', 'QB17', 'QB18', 'QB19', 'QB20', 'QB21', 'QB22', 'QB23', 'QB24']
Gates: dict_keys(['measure', 'measure_fidelity', 'prx', 'prx_12', 'cz', 'move', 'reset_wait', 'cc_prx'])


# Define Quantum Circuit

In [16]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(15, 5)
for i in range(15):
    qc.h(i)
for i in range(14):
    qc.cx(i, i + 1)
qc.cx(0, 14)
qc.measure(range(5), range(5))

# Transpile Circuit

In [17]:
from qaas.client.backend_iqm import transpile_to_IQM

# Transpile circuit Locally
qc_transpiled_local = transpile_to_IQM(qc, backend, optimize_single_qubits=False)

# Run Circuit

In [19]:
from time import time

# Run circuit with number of SHOTS
SHOTS = 100

client_execution_time = time()

# Passing as argument not transpiled circuit
job: QJob = backend.run([qc_transpiled_local], shots=SHOTS)
result = job.result()
client_execution_time = time() - client_execution_time
results_dict = result.get_counts()
print("Raw counts:", results_dict)

# Print non-zero results
for key, count in results_dict.items():
    if count > 0:
        print(f"State '{key}': {count} counts")
        # Only convert to int if key is actually a binary string
        if all(c in "01" for c in key):
            print(f"  -> Decimal value: {int(key, 2)}")

Raw counts: {'10110': 1, '10001': 5, '01010': 3, '10011': 5, '00000': 7, '10000': 9, '11110': 2, '00001': 4, '00100': 5, '11001': 5, '00011': 4, '10100': 5, '11011': 2, '11000': 4, '01100': 2, '01111': 2, '01101': 2, '00110': 5, '01000': 6, '00010': 4, '11111': 3, '01011': 4, '00111': 2, '10111': 2, '01001': 2, '01110': 1, '10101': 2, '00101': 2}
State '10110': 1 counts
  -> Decimal value: 22
State '10001': 5 counts
  -> Decimal value: 17
State '01010': 3 counts
  -> Decimal value: 10
State '10011': 5 counts
  -> Decimal value: 19
State '00000': 7 counts
  -> Decimal value: 0
State '10000': 9 counts
  -> Decimal value: 16
State '11110': 2 counts
  -> Decimal value: 30
State '00001': 4 counts
  -> Decimal value: 1
State '00100': 5 counts
  -> Decimal value: 4
State '11001': 5 counts
  -> Decimal value: 25
State '00011': 4 counts
  -> Decimal value: 3
State '10100': 5 counts
  -> Decimal value: 20
State '11011': 2 counts
  -> Decimal value: 27
State '11000': 4 counts
  -> Decimal value: 

In [20]:
print(f"Circuit runtime {job.remote_iqm_client_job_runtime} s")
print(f"Real HW execution {job.remote_hw_runtime} s")
print(f"Overall runtime on remote {job.remote_backend_runtime} s")
print("-" * 40)

print(f"QProvider runtime ended in {job.qaas_runtime} s")
print(f"QProvider fetched outputs in {job.qaas_fetching_runtime} s")
print(f"Pure client execution time {client_execution_time} s")

Circuit runtime 5.541268348693848 s
Real HW execution 1.233446 s
Overall runtime on remote 5.78713846206665 s
----------------------------------------
QProvider runtime ended in 9.42094111442566 s
QProvider fetched outputs in 11.172770023345947 s
Pure client execution time 35.86276817321777 s


In [ ]:
job.job_id

In [ ]:
for entry in result.timeline:
    # if entry.status in ["execution_started", "execution_ended"]:
    print(entry)

In [ ]:
from qiskit.visualization import plot_histogram

# Plot histogram
plot_histogram(results_dict, figsize=(12, 6), bar_labels=False)  # hide text above bars